In [ ]:
# ======================================================
# STEP 2: Import dependencies
# ======================================================
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import string
import re
import seaborn as sns
import matplotlib.pyplot as plt

import nltk
# NLTK setup
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# import fasttext
# from gensim.models import FastText, Word2Vec
from torch import nn
import torch
from torch.utils.data import DataLoader, Dataset


Mounted at /content/drive


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
import pandas as pd

folder = "/content/drive/MyDrive/Mini"

# Loading 3 datasets separately
df_wel = pd.read_csv(f"{folder}/welfake_clean.csv")
df_net = pd.read_csv(f"{folder}/fakenewsnet_clean.csv")
df_pred = pd.read_csv(f"{folder}/news_clean.csv")

print(df_wel.shape, df_net.shape, df_pred.shape)


(72134, 3) (23196, 3) (6335, 3)


In [ ]:
print(df_net.columns)
print(df_wel.columns)
print(df_pred.columns)
print(df_net.head())
print(df_wel.head())
print(df_pred.head())

Index(['combined_text', 'label', 'text'], dtype='object')
Index(['combined_text', 'label', 'text'], dtype='object')
Index(['combined_text', 'label', 'text'], dtype='object')
                                       combined_text  label  \
0  BREAKING: First NFL Team Declares Bankruptcy O...      0   
1  Court Orders Obama To Pay $400 Million In Rest...      0   
2  UPDATE: Second Roy Moore Accuser Works For Mic...      0   
3         Oscar Pistorius Attempts To Commit Suicide      0   
4        Trump Votes For Death Penalty For Being Gay      0   

                                                text  
0  breaking first nfl team declares bankruptcy kn...  
1          court order obama pay million restitution  
2  update second roy moore accuser work michelle ...  
3             oscar pistorius attempt commit suicide  
4                       trump vote death penalty gay  
                                       combined_text  label  \
0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...  

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.1-py3-none-any.whl.metadata (10.0 kB)
Using cached pybind11-3.0.1-py3-none-any.whl (293 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4498212 sha256=0d300f650f3b656092c1eb0a49409dd61f7f38540cb751c481e3fc77be2dbd62
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
import fasttext
import numpy as np

ft_unsup = fasttext.load_model("/content/drive/MyDrive/Mini/cc.en.300.bin")
ft_sup   = fasttext.load_model("/content/drive/MyDrive/Mini/fasttext_supervised.bin")

print("✅ FastText unsupervised loaded")
print("✅ FastText supervised loaded")


✅ FastText unsupervised loaded
✅ FastText supervised loaded


In [ ]:
import numpy as np

MAX_LEN = 300
EMB_DIM = 300

def text_to_seq(text, ft_model):
    text = str(text)
    words = text.split()
    seq = []

    # Convert up to MAX_LEN words → vectors
    for w in words[:MAX_LEN]:
        seq.append(ft_model.get_word_vector(w))

    # Pad with zeros
    while len(seq) < MAX_LEN:
        seq.append(np.zeros(EMB_DIM))

    return np.array(seq)


In [ ]:
import numpy as np
import os

def build_and_save_sequences(df, name, ft_model, model_type):
    """
    df         : dataset (welfake, fakenewsnet, newspred)
    name       : name for filenames
    ft_model   : unsupervised or supervised FastText
    model_type : 'unsup' or 'sup'
    """

    save_path = f"{folder}/{name}_{model_type}_seq.npy"
    labels_path = f"{folder}/{name}_labels.npy"

    print(f"▶ Processing {name} ({model_type}) → {save_path}")

    sequences = []
    for i, t in enumerate(df["text"]):
        sequences.append(text_to_seq(t, ft_model))
        if i % 1000 == 0:
            print(f"   {i}/{len(df)} done")

    sequences = np.array(sequences, dtype=np.float32)
    np.save(save_path, sequences)
    np.save(labels_path, df["label"].values)

    print(f"✅ Saved {name} {model_type} sequences:", sequences.shape)
    return save_path


In [ ]:
build_and_save_sequences(df_wel,  "welfake",      ft_unsup, "unsup")
build_and_save_sequences(df_wel,  "welfake",      ft_sup,   "sup")


▶ Processing welfake (unsup) → /content/drive/MyDrive/Mini/welfake_unsup_seq.npy
   0/72134 done


KeyboardInterrupt: 

In [ ]:
build_and_save_sequences(df_net,  "fakenewsnet",  ft_unsup, "unsup")
build_and_save_sequences(df_net,  "fakenewsnet",  ft_sup,   "sup")


In [ ]:
build_and_save_sequences(df_pred, "newspred",     ft_unsup, "unsup")
build_and_save_sequences(df_pred, "newspred",     ft_sup,   "sup")
